# Env39 — Topological Street-Network Metrics with NetworkX and OSMnx

## 1. Purpose

This notebook calculates the first research metrics for Env39. It is intentionally limited to network topology: connections, junctions, cycles, vulnerability, paths, and a concise centrality set. The tested `metrics_topology` module performs the calculations; notebook cells explain and present them for later Env38-versus-Env39 comparison.

In [ ]:
ENVIRONMENT = "env39"
CANONICAL_RUN = "latest"

## 2. Selected canonical run

The Phase 7D adapter independently verifies the file SHA-256, scientific-content signature, topology signature, and successful Phase 7C manifest before any metric is calculated. Editable and baseline files are not analysis inputs.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
PROJECT_ROOT = next((candidate for candidate in (start, *start.parents)
                     if (candidate / "environment.yml").is_file()
                     and (candidate / "src" / "geogami_morphology").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Repository root not found. Start Jupyter from the repository or notebooks directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from IPython.display import HTML, display
from geogami_morphology.metrics_topology import analyze_topology, load_topology_config
from geogami_morphology.notebook import degree_count_svg

result = analyze_topology(ENVIRONMENT, canonical_run=CANONICAL_RUN, project_root=PROJECT_ROOT, publish=False)
config = load_topology_config(project_root=PROJECT_ROOT)
metric = result.calculation.metric
print("REPOSITORY_DISCOVERY: PASS")
print("Canonical run:", result.selection.run_id)
print("Canonical file SHA-256:", result.selection.file_sha256)
print("Scientific-content signature:", result.selection.scientific_content_signature)
print("Topology signature:", result.selection.topology_signature)
print("CANONICAL_RUN_RESOLUTION: PASS")

## 3. Scientific graph representations

`G_physical` is an undirected NetworkX `MultiGraph`: one edge means one physical canonical street. `G_ox` is OSMnx-compatible and has two reciprocal arcs per street. Algorithms needing a simple graph use `G_simple` only after the module proves that there are no self-loops or parallel physical edges and that node, edge, endpoint, and canonical edge identities are unchanged.

In [ ]:
print(f"G_physical: {result.physical_graph.number_of_nodes()} nodes / {result.physical_graph.number_of_edges()} physical streets")
print(f"G_ox: {result.osmnx_graph.number_of_nodes()} nodes / {result.osmnx_graph.number_of_edges()} reciprocal directed arcs")
print(f"G_simple: {result.calculation.simple_graph.number_of_nodes()} nodes / {result.calculation.simple_graph.number_of_edges()} physical streets")
assert result.physical_graph.number_of_edges() == 69
assert result.osmnx_graph.number_of_edges() == 138
print("GRAPH_REPRESENTATIONS: PASS")

## 4. Metric configuration

Configuration fixes the authoritative `length` cost, local-unit semantics, OSMnx intersection threshold, disconnected-network policy, and betweenness normalization. Its SHA-256 is stored with the analysis provenance.

In [ ]:
print("Configuration schema:", config.schema_version)
print("Configuration SHA-256:", config.file_sha256)
print("Weighted distance attribute / units: length / local units")
print("Intersection minimum physical streets:", config.osmnx_intersection_min_streets)
print("METRIC_CONFIGURATION: PASS")

## 5. Network identity

Identity metrics check the size and connectedness of the physical topology. The handshake theorem independently compares the sum of node degrees with twice the physical edge count; reciprocal OSMnx arcs are never substituted for physical streets.

In [ ]:
identity_names = ["node_count", "physical_edge_count", "connected_component_count", "is_connected", "self_loop_count", "parallel_physical_edge_count", "degree_sum", "twice_physical_edge_count"]
display(pd.DataFrame([{"metric": name, "value": metric(name)} for name in identity_names]))
assert metric("degree_sum") == metric("twice_physical_edge_count") == 138
print("NETWORK_IDENTITY_AND_HANDSHAKE: PASS")

## 6. Degree and junction structure

Node degree measures how many physical streets meet at a canonical node. It is unweighted and dimensionless. Dead ends have degree 1; degree-3 and degree-4 nodes describe common junction forms useful in comparing access structure between environments.

In [ ]:
degree_names = ["degree_min", "degree_max", "degree_mean", "degree_median", "dead_end_count", "dead_end_proportion", "degree_3_count", "degree_3_proportion", "degree_4_count", "degree_4_proportion"]
display(pd.DataFrame([{"metric": name, "value": metric(name)} for name in degree_names]))
degree_counts = {int(key): value for key, value in result.calculation.osmnx_crosscheck["streets_per_node_counts"].items() if value}
assert degree_counts == {1: 10, 3: 16, 4: 20}
assert metric("degree_mean") == 3.0
print("DEGREE_STRUCTURE: PASS")

## 7. NetworkX / OSMnx street-count cross-check

OSMnx 2.1.1 converts the reciprocal representation to an undirected view to count physical streets. The module explicitly populates `street_count`, then checks every node against NetworkX physical degree. With `min_streets = 2`, an intersection is any node incident to at least two physical streets.

In [ ]:
cross = result.calculation.osmnx_crosscheck
assert all(row["networkx_degree"] == row["osmnx_street_count"] for row in cross["node_by_node"])
display(pd.DataFrame(cross["node_by_node"]).head(10))
print("OSMnx average streets/node:", cross["streets_per_node_average"])
print("OSMnx intersections (min_streets=2):", cross["intersection_count"])
print("OSMnx physical segments / directed arcs:", cross["physical_street_segment_count"], "/", cross["directed_arc_count_not_physical_streets"])
print("OSMNX_NODE_BY_NODE_CROSSCHECK: PASS")

## 8. Connectedness and cycle rank

The cyclomatic number `E - V + C` counts independent topological cycles. Node and edge connectivity give the minimum number of nodes or physical streets whose removal can disconnect the current connected graph. These are unweighted `G_simple` measures.

In [ ]:
connectivity_names = ["cycle_rank", "node_connectivity", "edge_connectivity"]
display(pd.DataFrame([{"metric": name, "value": metric(name)} for name in connectivity_names]))
assert metric("cycle_rank") == 24
print("CONNECTEDNESS_AND_CYCLE_RANK: PASS")

## 9. Bridges and articulation points

A bridge is a physical street whose removal increases the number of components. An articulation point is a node with the analogous property. Canonical `edge_id` and `node_id` values make the vulnerability results traceable back to the accepted network.

In [ ]:
nodes_frame = pd.DataFrame(result.calculation.node_rows)
edges_frame = pd.DataFrame(result.calculation.edge_rows)
display(pd.DataFrame([{"metric": name, "value": metric(name)} for name in ["bridge_count", "bridge_proportion", "articulation_point_count", "articulation_point_proportion"]]))
display(edges_frame.loc[edges_frame.is_bridge, ["edge_id", "u", "v"]])
display(nodes_frame.loc[nodes_frame.is_articulation_point, ["node_id", "degree"]])

## 10. Shortest-path structure

Hop paths assign every physical street cost 1 and report results in hops. Weighted paths assign the authoritative geometry-derived `length` as cost and therefore report local units, never metres. No geographic distance routine is used.

In [ ]:
path_names = ["average_shortest_path_hops", "diameter_hops", "radius_hops", "average_shortest_path_length_local"]
display(pd.DataFrame([{"metric": name, "value": metric(name)} for name in path_names]))
print("WEIGHTED_PATH_UNITS: local units")

## 11. Centrality

Degree centrality normalizes degree by `V - 1`. Betweenness measures how often a node or street lies on shortest paths; its weighted form treats `length` as distance cost. Closeness is the reciprocal of average shortest-path distance, so weighted closeness has units `1 / local unit` and larger values mean shorter access distances.

In [ ]:
display(nodes_frame.sort_values("betweenness_length", ascending=False)[["node_id", "degree_centrality", "betweenness_unweighted", "betweenness_length", "closeness_unweighted", "closeness_length"]].head(10))
display(edges_frame.sort_values("betweenness_length", ascending=False)[["edge_id", "betweenness_unweighted", "betweenness_length"]].head(10))

## 12. Compact result table

The long-form summary records value, units, library, function, graph representation, weighting, and interpretation rather than hiding structured values inside opaque CSV cells.

In [ ]:
summary = pd.DataFrame(result.calculation.summary_rows)
display(summary[["metric_name", "value", "units", "graph_representation", "weighting"]])

## 13. Minimal figures

One deterministic inline SVG shows the degree distribution. It has no GUI backend, no embedded binary image, and no Phase 7G geometry or orientation content.

In [ ]:
display(HTML(degree_count_svg(degree_counts)))
print("HEADLESS_DEGREE_FIGURE: PASS")

## 14. Exported artifacts

The command-line workflow publishes stable summary, node, edge, OSMnx cross-check, methodology, and complementary analysis-manifest files under the selected run's `topology/` directory. This notebook calculates in memory and verifies those already-published artifacts without overwriting Phase 7D provenance.

In [ ]:
required = ["topology_metrics_summary.csv", "topology_metrics_summary.json", "node_topology_metrics.csv", "edge_topology_metrics.csv", "osmnx_crosscheck.json", "topology_methodology.json", "topology_analysis_manifest.json"]
missing = [name for name in required if not (result.output_dir / name).is_file()]
if missing:
    raise RuntimeError(f"Missing published topology artifacts: {missing}. Run scripts/run_topology_metrics.py first.")
print("Artifacts directory:", result.output_dir.relative_to(PROJECT_ROOT))
print("EXPORTED_ARTIFACTS: PASS")

## 15. Interpretation

Phase 7F describes how Env39 is connected, where its junctions and topological vulnerabilities lie, and how its minimum paths distribute across nodes and streets. These quantities support later controlled Env38-versus-Env39 comparison. They do not describe street straightness, curvature, orientation, entropy, or rose-diagram structure; those geometric morphology questions remain for Phase 7G.

## 16. Final reproducibility summary

The selected canonical identities, versioned metric configuration, library versions, node-by-node OSMnx cross-check, stable schemas, and artifact hashes together make this analysis auditable. The selected canonical manifest's original Git dirty state remains visible in `topology_analysis_manifest.json`.

In [ ]:
assert (metric("node_count"), metric("physical_edge_count"), metric("connected_component_count")) == (46, 69, 1)
assert result.selection.topology_signature == "4AF93910532EB430103267DAECF73B3ADC3B3D1599471EDF6B1632B6C5F898CB"
print("Geographic distance/bearing functions: NOT USED")
print("Phase 7G morphology metrics: NOT CALCULATED")
print("NOTEBOOK_02_EXECUTION: PASS")